# Step 5: Persiapan Dataset untuk Training

Menggabungkan fitur (Step 3) + label (Step 4) menjadi dataset final:
1. Merge `features.csv` + `labeled_boards.csv`
2. Normalisasi nama kolom (jika menggunakan fallback Bridge_Prediction)
3. Filter board yang valid (ada label DDS, bukan pass-out)
4. Pilih kolom fitur sesuai C23 Tabel 2
5. Split dataset → train (80%) dan test (20%)

**Prasyarat:** Jalankan `03_features.ipynb` dan `04_labeling.ipynb` terlebih dahulu.

In [ ]:
import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

DATA_PROCESSED = Path(ROOT) / "data" / "processed"
RESULTS        = Path(ROOT) / "results"

FEATURES_CSV = DATA_PROCESSED / "features.csv"
LABELED_CSV  = DATA_PROCESSED / "labeled_boards.csv"
DATASET_CSV  = DATA_PROCESSED / "bridge_dataset.csv"

# Fallback jika DDS tidak tersedia
BRIDGE_PRED_CSV = Path("d:/Bridge_Prediction/data/processed/bridge_features.csv")

print(f"Root proyek : {ROOT}")
print("Setup selesai.")

## 5.1 Merge Fitur + Label

In [ ]:
from features import FEATURE_COLS

DDS_AVAILABLE = LABELED_CSV.exists() and "labeled_boards" in str(LABELED_CSV)

if FEATURES_CSV.exists() and LABELED_CSV.exists():
    df_features = pd.read_csv(FEATURES_CSV)
    print(f"Features loaded: {df_features.shape}")

    label_cols = ["source_file", "board_number",
                  "best_contract_strain", "best_contract_category",
                  "best_contract_level", "best_contract_token"]
    df_lbl_all = pd.read_csv(LABELED_CSV)
    avail_label_cols = [c for c in label_cols if c in df_lbl_all.columns]
    df_lbl = df_lbl_all[avail_label_cols]

    merge_keys = [k for k in ["source_file", "board_number"] if k in df_features.columns and k in df_lbl.columns]
    if merge_keys:
        df_dataset = df_features.merge(df_lbl, on=merge_keys, how="inner")
        print(f"Labeled loaded  : {df_lbl.shape}")
        print(f"Setelah merge   : {df_dataset.shape}")
    else:
        print("Tidak ada key merge yang cocok — menggunakan labeled_boards langsung")
        df_dataset = df_lbl_all

elif BRIDGE_PRED_CSV.exists():
    print(f"Menggunakan fallback: {BRIDGE_PRED_CSV}")
    df_dataset = pd.read_csv(BRIDGE_PRED_CSV)
    print(f"Shape: {df_dataset.shape}")

else:
    raise FileNotFoundError(
        "Tidak ditemukan features.csv + labeled_boards.csv maupun bridge_features.csv. "
        "Jalankan Step 3 dan Step 4 terlebih dahulu."
    )

print(f"\nKolom total: {len(df_dataset.columns)}")

## 5.2 Normalisasi Nama Kolom

Jika menggunakan fallback `bridge_features.csv` dari Bridge_Prediction, nama kolomnya berbeda dari `FEATURE_COLS`. Fungsi `normalize_columns` memetakan nama tersebut.

In [ ]:
def normalize_columns(df):
    """Rename kolom bridge_features.csv agar cocok dengan FEATURE_COLS."""
    rename = {}
    for suit in ["spade", "heart", "diamond", "club"]:
        if f"N_hcp_{suit}" in df.columns:
            rename[f"N_hcp_{suit}"] = f"player_hcp_{suit}"
        if f"S_hcp_{suit}" in df.columns:
            rename[f"S_hcp_{suit}"] = f"partner_hcp_{suit}"
        if f"NS_num_{suit}" in df.columns:
            rename[f"NS_num_{suit}"] = f"total_num_{suit}"
        if f"NS_total_stop_{suit}" in df.columns:
            rename[f"NS_total_stop_{suit}"] = f"total_stop_{suit}"
    if "NS_total_hcp" in df.columns:
        rename["NS_total_hcp"] = "total_hcp"
    if "N_balance" in df.columns:
        rename["N_balance"] = "player_balance"
    if "S_balance" in df.columns:
        rename["S_balance"] = "partner_balance"
    if "vulnerability_int" in df.columns:
        rename["vulnerability_int"] = "vulnerability"
    df = df.rename(columns=rename)

    # Buat dealer one-hot jika belum ada
    if "board_dealer" in df.columns and "dealer_N" not in df.columns:
        for seat in ["N", "E", "S", "W"]:
            df[f"dealer_{seat}"] = (df["board_dealer"] == seat).astype(int)

    # ns_best_fit dari total_num_* jika belum ada
    num_cols = [f"total_num_{s}" for s in ["spade", "heart", "diamond", "club"]]
    if "ns_best_fit" not in df.columns and all(c in df.columns for c in num_cols):
        df["ns_best_fit"] = df[num_cols].max(axis=1)
        df["ns_best_major_fit"] = df[["total_num_spade", "total_num_heart"]].max(axis=1)
    return df

df_model = df_dataset.dropna(subset=["best_contract_strain", "best_contract_category"])
df_model = normalize_columns(df_model.copy())
print(f"Setelah drop NA + normalize kolom: {len(df_model)} baris")

## 5.3 Persiapan Feature Matrix dan Target Label

In [ ]:
from model import prepare_features

available_feat = [c for c in FEATURE_COLS if c in df_model.columns]
missing_feat   = [c for c in FEATURE_COLS if c not in df_model.columns]

print(f"Fitur tersedia : {len(available_feat)}/{len(FEATURE_COLS)}")
if missing_feat:
    print(f"Fitur tidak ada: {missing_feat[:5]}{'...' if len(missing_feat) > 5 else ''}")

X      = prepare_features(df_model, available_feat)
y_suit = df_model["best_contract_strain"]
y_cat  = df_model["best_contract_category"]

print(f"\nFeature matrix X : {X.shape}")
print(f"Label suit       : {y_suit.value_counts().to_dict()}")
print(f"Label kategori   : {y_cat.value_counts().to_dict()}")

## 5.4 Simpan Dataset Final

In [ ]:
df_model.to_csv(DATASET_CSV, index=False)
print(f"Dataset final disimpan ke: {DATASET_CSV}")
print(f"Shape: {df_model.shape}")

## 5.5 Split Train / Test (80/20 Stratified)

In [ ]:
X_train, X_test, y_suit_train, y_suit_test = train_test_split(
    X, y_suit, test_size=0.2, stratify=y_suit, random_state=42
)
_, _, y_cat_train, y_cat_test = train_test_split(
    X, y_cat, test_size=0.2, stratify=y_suit, random_state=42
)

print(f"Train set : {X_train.shape[0]} sampel ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set  : {X_test.shape[0]} sampel ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"Fitur     : {X_train.shape[1]}")
print(f"\nDistribusi suit di train:")
print(y_suit_train.value_counts().to_string())
print(f"\nDistribusi suit di test:")
print(y_suit_test.value_counts().to_string())

In [ ]:
# Simpan split index untuk reproducibility di notebook berikutnya
import joblib

split_data = {
    "X_train": X_train, "X_test": X_test,
    "y_suit_train": y_suit_train, "y_suit_test": y_suit_test,
    "y_cat_train": y_cat_train,  "y_cat_test": y_cat_test,
}
SPLIT_PATH = RESULTS / "metrics" / "dataset_split.pkl"
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(split_data, SPLIT_PATH)
print(f"Split data disimpan ke: {SPLIT_PATH}")

---
## Output

File yang dihasilkan:
- `data/processed/bridge_dataset.csv` — dataset final siap training
- `results/metrics/dataset_split.pkl` — train/test split tersimpan

**Langkah berikutnya:** Buka `06_training.ipynb` untuk melatih model.